# Model Routing

## Scenario: Northstar chooses the smallest eligible model path

A support and incident assistant must format a known status, investigate a regional regression, interpret a dashboard screenshot, and prepare a tested code patch. We will route each task by capability, quality/risk floor, latency SLO, and cost budget—not by a model's self-confidence alone.

**Safety boundary:** every route is proposal-only. Routing never authorizes tools, production actions, customer contact, or a merge.

**Learning outcomes:** distinguish capability, cost, and latency routing; build bounded cascades and fallbacks; decide when ensembles earn their cost; and evaluate a routing policy rather than only an answer.

![Policy-bound model routing lifecycle](../../../assets/model-routing-policy.svg)

A router first eliminates ineligible choices (wrong modality, unavailable, policy-disallowed), then selects among eligible routes. An evaluator decides whether to accept, promote, abstain, or request human review. This is different from internal mixture-of-experts routing: this lesson routes *whole deployed model paths*.

## 1. Build a model catalog and task contract

A production catalog is versioned and monitored: model capability, supported modalities, structured-output/tool features, data residency, price, measured task-family quality, p95 latency, availability, and deprecation date. A task contract carries only trustworthy application-owned fields such as modality, complexity, risk, latency SLO, and whether code is required.

**When to use which approach:** choose a small/fast model for a tested known path; a reasoning path for ambiguous multi-step evidence synthesis; a multimodal path when visual/document evidence is necessary; and a coding path for repository/test/diff work. No route is automatically suitable merely because it is more expensive.

In [1]:
from pathlib import Path
import sys

TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'advanced' / '09-model-routing'
sys.path.insert(0, str(TOPIC))
from lab import Task, choose_route, run_cascade, select_ensemble

tasks = [
    Task('format EU status'),
    Task('investigate regional checkout regression', complexity=9, risk='high'),
    Task('read dashboard screenshot', modality='screen'),
    Task('prepare retry patch with tests', needs_code=True),
]
for task in tasks:
    route = choose_route(task)
    print(f'{task.name:42} -> {route.model:12} | {route.reason}')

assert [choose_route(task).model for task in tasks] == ['fast-text', 'reasoning', 'multimodal', 'coding']

format EU status                           -> fast-text    | known, text-only path
investigate regional checkout regression   -> reasoning    | ambiguous, multi-step evidence synthesis
read dashboard screenshot                  -> multimodal   | image, screen, or document understanding
prepare retry patch with tests             -> coding       | repository, patch, or test-oriented task


## 2. Capability routing before price routing

The screenshot cannot silently become a text-only task; the patch cannot be sent to a generic route just because it is cheaper. Capability is a hard eligibility filter. After filtering, cost routing chooses the lowest expected-cost route that meets a *measured* task-family quality floor. Latency routing adds a deadline: account for router time, queueing, retry/promotion budget, and p95—not only a provider's median model time.

A useful release objective is: `minimize expected cost subject to quality >= floor, p95 latency <= SLO, and policy = allowed`. Track cost per successful task, not cost per call.

In [2]:
# Deliberate failure: the cheap path produces weak externally measured support.
# A bounded cascade promotes once; it does not keep retrying until it likes an answer.
cascade = run_cascade(Task('summarize a known incident'), quality_signal=0.52)
print('cascade:', ' -> '.join(route.model for route in cascade))
assert [route.model for route in cascade] == ['fast-text', 'reasoning']

# A required capability outage should fail safely instead of pretending text can see a screen.
unavailable = choose_route(Task('inspect payment screenshot', modality='screen'), available={'fast-text', 'reasoning'})
print('unavailable capability:', unavailable.model, '|', unavailable.reason)
assert unavailable.model == 'human-review'

cascade: fast-text -> reasoning
unavailable capability: human-review | required multimodal capability unavailable


## 3. Cascades, fallbacks, and ensembles

A **cascade** tries a cheaper eligible route, then promotes after an acceptance test fails. Good acceptance signals are schema validation, tests, citations, deterministic rules, calibrated task-specific evaluators, or sampled human review. A **fallback** is selected because the preferred path is unavailable, times out, or cannot meet its contract. Do not label a quality downgrade as a transparent fallback for a high-risk task.

An **ensemble** obtains independently useful candidate outputs and uses a verifier or a human to select/reject. Use it for high-value uncertainty with genuine model/evidence diversity. It costs more, increases latency, and does not turn a majority vote into truth. Shared poisoned context creates correlated failure, so all candidates still need the same provenance, tenant, and permission checks.

In [3]:
high_risk = Task('recommend mitigation for EU conversion drop', complexity=9, risk='high')
print('ensemble policy:', select_ensemble(high_risk, disagreement=0.40))
print('routine policy:', select_ensemble(Task('format status'), disagreement=0.40))
assert select_ensemble(high_risk, 0.40) == 'independent-models-plus-verifier'

# A compact routing-evaluation record. In production, join this with answer quality,
# provenance, user correction, provider health, queue time, and catalog/policy versions.
record = {
    'task_family': 'incident_synthesis', 'route': 'reasoning', 'route_eligible': True,
    'accepted': True, 'quality_score': 0.91, 'latency_ms': 3100, 'cost_cents': 1.20,
    'promotion_count': 0, 'policy_version': '2026-08-10',
}
cost_per_success = record['cost_cents'] / int(record['accepted'])
print(record, 'cost_per_success=', cost_per_success)

ensemble policy: independent-models-plus-verifier
routine policy: single-route-with-evaluation
{'task_family': 'incident_synthesis', 'route': 'reasoning', 'route_eligible': True, 'accepted': True, 'quality_score': 0.91, 'latency_ms': 3100, 'cost_cents': 1.2, 'promotion_count': 0, 'policy_version': '2026-08-10'} cost_per_success= 1.2


## 4. Production checklist and exercises

**Before release:** evaluate route correctness and final quality per task family, modality, language, risk tier, and tenant policy; test timeout, outage, malformed output, wrong-modality, exhausted cascade, and ensemble-disagreement paths; version/audit catalog and policy decisions; alert on quality-floor violation and fallback spikes; retain a last-known-good policy; and keep identity, data boundaries, approval, idempotency, budgets, and tool authorization outside the router.

**Exercises**

1. Add a maximum-cost field. Why should a high-risk task return `human-review` rather than violate its quality floor?
2. Implement an acceptance check that requires two source IDs for a grounded answer. Compare always-reasoning with the cascade by cost per successful task.
3. Introduce a multimodal outage. Decide what can be deferred and why text-only guessing is unsafe.
4. Build a held-out routing set where average quality conceals poor performance for one language or high-value tenant. Define the release gate.

**Further reading:** [RouteLLM](https://arxiv.org/abs/2406.18665), [RouterBench](https://arxiv.org/abs/2403.12031), [FrugalGPT](https://arxiv.org/abs/2305.05176), and [Unified routing and cascading](https://arxiv.org/abs/2410.10347).